In [ ]:
import torch
import torch.nn as nn
import numpy as np
from xgboost import XGBClassifier
import os
import pandas as pd
from glob import glob
import rasterio as rio
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from src.mslandcover.models import HRNetSegmentationModel
from src.mslandcover.config import HRNET_BASE_CONFIG

In [135]:
target_paths = glob('data/png_images/batch_0/target/*.tif')
input_paths = [x.replace('target', 'input_tif') for x in target_paths]

img_paths_df = pd.DataFrame({'input': input_paths, 'target': target_paths})
img_paths_df['X'] = img_paths_df['input'].apply(lambda x: rio.open(x).read())
img_paths_df['y'] = img_paths_df['target'].apply(lambda x: rio.open(x).read())

In [3]:
hrnet_w18_config = {
    'STAGE1': {
        'NUM_MODULES': 1,
        'NUM_BRANCHES': 1,
        'BLOCK': 'BOTTLENECK',
        'NUM_BLOCKS': [4],
        'NUM_CHANNELS': [64],
        'FUSE_METHOD': 'SUM',
    },
    'STAGE2': {
        'NUM_MODULES': 1,
        'NUM_BRANCHES': 2,
        'BLOCK': 'BASIC',
        'NUM_BLOCKS': [4, 4],
        'NUM_CHANNELS': [18, 36],
        'FUSE_METHOD': 'SUM',
    },
    'STAGE3': {
        'NUM_MODULES': 4,
        'NUM_BRANCHES': 3,
        'BLOCK': 'BASIC',
        'NUM_BLOCKS': [4, 4, 4],
        'NUM_CHANNELS': [18, 36, 72],
        'FUSE_METHOD': 'SUM',
    },
    'STAGE4': {
        'NUM_MODULES': 3,
        'NUM_BRANCHES': 4,
        'BLOCK': 'BASIC',
        'NUM_BLOCKS': [4, 4, 4, 4],
        'NUM_CHANNELS': [18, 36, 72, 144],
        'FUSE_METHOD': 'SUM',
    },
    'IMAGE_DECODER': {
        'NUM_BLOCKS': 2, # number of blocks per decoder layer - 2 is the default for simplicity
    },
    'SIMCLR_PROJECTION_HEAD': {
        'NUM_HIDDENS': 1, # number of hidden layers to use in the projection head
        'EMBED_DIM': 128,
    }
}

In [137]:
X_batch = np.stack(img_paths_df['X'].values)
X_batch = torch.from_numpy(X_batch).float()

model = HRNetSegmentationModel(hrnet_w18_config)
# remove incre_modules in encoder
# model.encoder.incre_modules = nn.Identity()

# remove decoder
model.decoder = nn.Identity()
model.img_decoder_activation = nn.Identity()

# remove projection head
model.projection_head = nn.Identity()
model.load_state_dict(torch.load('./weights/hrnet_w18/hsv_simclr_old.pth'), strict=False)
    
print(model)
# model.load_state_dict(torch.load('./weights/hrnet_w18/hsv_simclr_old.pth', weights_only=True), strict=False)

C:\Users\dh2306\AppData\Local\Temp\ipykernel_53056\2172633861.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('./weights/hrnet_w18/hsv_

HRNetSegmentationModel(
  (encoder): HighResolutionNet(
    (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, 

In [138]:
model.eval()
model = model.to('cuda')

# print(X_batch.shape)

with torch.no_grad():
    X_batch = X_batch.to('cuda')
    y_pred = model(X_batch)[0] # only return the first output - projection head is missing
    
    y_pred = nn.functional.interpolate(y_pred, scale_factor=4, mode='bilinear', align_corners=True)

print(y_pred.shape)
img_paths_df['z'] = list(y_pred.cpu().numpy())

torch.Size([55, 270, 256, 256])


In [114]:
print(img_paths_df['z'].values.shape)
print(len(img_paths_df['X']))
print(len(img_paths_df['y']))  

(55,)
55
55


In [139]:
n_features = img_paths_df['z'].values[0].shape[0]
print(n_features)

270


In [189]:
# add hand-crafted features
def caculate_ndvi(X):
    # nir band is 0
    # red band is 1
    X = X.astype(np.float32)
    return (X[0] - X[1]) / (X[0] + X[1])

def caculate_ndwi(X):
    # nir band is 0
    # green band is 2
    X = X.astype(np.float32)
    return (X[2] - X[0]) / (X[2] + X[0])

def caclulate_gndvi(X):
    # nir band is 0
    # green band is 2
    X = X.astype(np.float32)
    return (X[0] - X[2]) / (X[0] + X[2])



img_paths_df['ndvi'] = img_paths_df['X'].apply(caculate_ndvi)
img_paths_df['ndwi'] = img_paths_df['X'].apply(caculate_ndwi)
img_paths_df['gndvi'] = img_paths_df['X'].apply(caclulate_gndvi)

# append the hand-crafted features to the model output
agg_features = np.stack(img_paths_df['z'].values)

print(np.stack(img_paths_df['X'].values)[:,0][:, None].shape)
print(np.stack(img_paths_df['ndvi'].values)[:, None].shape)

# print(X.shape)
agg_features = np.concatenate([
    agg_features, 
    np.stack(img_paths_df['ndvi'].values)[:, None], 
    np.stack(img_paths_df['ndwi'].values)[:, None], 
    np.stack(img_paths_df['gndvi'].values)[:, None], 
    np.stack(img_paths_df['X'].values)[:,0][:, None],
    np.stack(img_paths_df['X'].values)[:,1][:, None],
    np.stack(img_paths_df['X'].values)[:,2][:, None]],
axis=1)
n_features = agg_features.shape[1]

img_paths_df['agg_features'] = list(agg_features)

C:\Users\dh2306\AppData\Local\Temp\ipykernel_53056\3432741601.py:6: RuntimeWarning: invalid value encountered in divide
  return (X[0] - X[1]) / (X[0] + X[1])
C:\Users\dh2306\AppData\Local\Temp\ipykernel_53056\3432741601.py:12: RuntimeWarning: invalid value encountered in divide
  return (X[2] - X[0]) / (X[2] + X[0])
C:\Users\dh2306\AppData\Local\Temp\ipykernel_53056\3432741601.py:18: RuntimeWarning: invalid value encountered in divide
  return (X[0] - X[2]) / (X[0] + X[2])


(55, 1, 256, 256)
(55, 1, 256, 256)


In [190]:
train_df, test_df = train_test_split(img_paths_df, test_size=0.2)

offset = (256 - 192) // 2 # only keep the center 192x192 pixels due to edge effects

# print(np.stack(.shape)

X_train = np.stack(train_df['agg_features'])[:, :, offset:-offset, offset:-offset].transpose(0, 2, 3, 1).reshape(-1, n_features)
y_train = np.stack(train_df['y'].values)[:, :, offset:-offset, offset:-offset].flatten()

X_test = np.stack(test_df['agg_features'])[:, :, offset:-offset, offset:-offset].transpose(0, 2, 3, 1).reshape(-1, n_features)
y_test = np.stack(test_df['y'].values)[:, :, offset:-offset, offset:-offset].flatten()

# X_test = np.stack([z.transpose(1, 2, 0) for z in test_df['z'].values])
# X_test = X_test.reshape(-1, 720)
# y_test = np.stack([y.transpose(1, 2, 0) for y in test_df['y'].values])
# y_test = y_test.reshape(-1, 1)

In [96]:
print(len(X_train), len(y_train))

1622016 1622016


In [191]:
# print distributio of classes
print(np.unique(y_train, return_counts=True))

# remove 0 class from the data
X_train = X_train[y_train != 0]
y_train = y_train[y_train != 0] - 1

X_test = X_test[y_test != 0]
y_test = y_test[y_test != 0] - 1

from imblearn.over_sampling import SMOTE

oversample = SMOTE()
X_train, y_train = oversample.fit_resample(X_train, y_train)

# print(np.unique(y_train, return_counts=True))



(array([0, 1, 2, 3, 4, 5, 6, 7, 8], dtype=uint8), array([ 73963,  66675,  22623,  55516, 115075, 917502, 260410,  42681,
        67571]))


In [ ]:
model = XGBClassifier(n_estimators=100, n_jobs=-1)

model.fit(X_train, y_train)

print(model.score(X_test, y_test))

y_preds = model.predict(X_test)
print(classification_report(y_test, y_preds))

In [ ]:
fis = model.feature_importances_
sorted_f1s = np.argsort(fis)[::-1]
print(sorted_f1s)

[270 274  46 126 272 208 271 273 129 243 253  66 213 239 227 246 247  64
 254 248 200  65 275 230 156  82 171  97 149  85 269 148 104  22   8 151
 137 112 225  39 184 194 167  47  56 100 175 187  95  40 143 262 220 116
  92 211 202 249  32 228 135 209 237 152 174  58 123  62 186 158 207 236
 210 223 267 206  72 205 183 166 199 177 233 265 164 105 145 242 141 189
 103 180 259 179 144  74  99 139  51  94 240 221 170 241 229  43 251 160
 155 196 257 234 204 130 178  80  88  55 132 182 102  76 192 218 146 216
  91 232 263  50 172 121  52  73 136 133 203 250  60 138  25 260 191 190
 176  98  13  28  84 258  77 128 235  41 193 163 150  35 109  11  93  27
 231  19  15 224 219  49 161 154  90 226  86 222 114 142  26  68  20 119
 115 127 106 157 215  38 120  53   0  83 168  31 140  24  42 124  17  89
 117  45 244  69  14  54  44 131 110 268 134  23 173  57   3 256   1 162
 169 125  79  10   5 198  71  36  12 261 266 264 153 159   4  18  16   9
   7   6 165 147 122 118 195 188 185 181  75  78  8

In [ ]:
X_train_spectral = X_train[:, n_features-6:]
X_test_spectral = X_test[:, n_features-6:]

model_spectral = XGBClassifier(n_estimators=100, n_jobs=-1)

model_spectral.fit(X_train_spectral, y_train)

print(model_spectral.score(X_test_spectral, y_test))

y_preds_spectral = model_spectral.predict(X_test_spectral)

print(classification_report(y_test, y_preds_spectral))

0.6744495738636364
              precision    recall  f1-score   support

           0       0.05      0.01      0.01     18007
           1       1.00      0.96      0.98     45597
           2       0.34      0.46      0.39      5910
           3       0.62      0.25      0.36     15736
           4       0.04      0.25      0.07      2248
           5       0.65      0.95      0.77    155185
           6       0.75      0.62      0.68    114836
           7       0.02      0.00      0.00     35848
           8       0.42      0.34      0.37     12137

    accuracy                           0.67    405504
   macro avg       0.43      0.43      0.41    405504
weighted avg       0.62      0.67      0.63    405504



In [181]:
from sklearn.svm import SVC
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

model2 = make_pipeline(
    RobustScaler(),
    LogisticRegression(max_iter=10000, n_jobs=-1)
)

model2.fit(X_train, y_train)

print(model2.score(X_test, y_test))

y_preds = model2.predict(X_test)

print(classification_report(y_test, y_preds))


KeyboardInterrupt: 